# SS3 Part B — STADIOEquities Account Activation


## Project aim

This notebook applies the Random Forest method selected in SS2
to the supplied synthetic STADIOEquities client data.

The business question is:

Among customers who have not activated three days after registration,
can information available at that point predict activation by day 30?

Activation means making a successful first deposit of at least R100
within 30 calendar days of registration.

- Class 0: Not activated within 30 days.
- Class 1: Activated within 30 days.

Low predicted activation probabilities could help prioritise customers
for follow-up. Predictions do not establish whether contacting customers
will cause them to activate.

This notebook examines activation, because the extract does not provide
a suitable six-month dormancy outcome.

In [22]:
import os
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from sklearn.calibration import calibration_curve

RANDOM_STATE = 42
LANDMARK_DAYS = 3
HORIZON_DAYS = 30

print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Dataset loading

Five datasets are supplied:

1. Registration, KYC and customer information.
2. App and web behaviour.
3. Account funding and activation outcomes.
4. Marketing activity.
5. Data-quality metadata.

The raw registration dataset contains 1,898 rows, including duplicates.
The funding dataset contains 1,893 customers.

The supplied funding labels contain:
- 1,085 non-activated customers: 57.32%.
- 808 activated customers: 42.68%.

These proportions indicate moderate class imbalance.

Behaviour and marketing datasets contain multiple records per customer.
They will be aggregated before joining to customer-level records.

The metadata dataset has no customer identifier, so it is used for
extract information and aggregate quality checks.

In [23]:
data_folder = (
    r"C:\Users\kone\OneDrive - Seriti Business Solutions, Pty (Ltd)"
    r"\Documents\STADIO\CAPS\Capstone_Project_KM"
    r"\Data\stadioequities\stadioequities"
)

results_folder = os.path.join(os.getcwd(), "results")
os.makedirs(results_folder, exist_ok=True)

if not os.path.isdir(data_folder):
    raise FileNotFoundError(
        f"Data folder not found:\n{data_folder}"
    )

print("Data folder found.")
print("\nFiles inside:")

for filename in os.listdir(data_folder):
    print(filename)

Data folder found.

Files inside:
DATA_DICTIONARY.md
README.md
source1_registration_kyc_customer.csv
source2_app_web_behaviour.csv
source3_account_funds_activation.csv
source4_marketing_engagement.csv
source5_data_quality_metadata.csv


In [24]:
files = {
    "registration": "source1_registration_kyc_customer.csv",
    "behaviour": "source2_app_web_behaviour.csv",
    "funding": "source3_account_funds_activation.csv",
    "marketing": "source4_marketing_engagement.csv",
    "metadata": "source5_data_quality_metadata.csv"
}

loaded_data = {}

for name, filename in files.items():
    file_path = os.path.join(data_folder, filename)

    if not os.path.isfile(file_path):
        raise FileNotFoundError(
            f"Required file not found:\n{file_path}"
        )

    loaded_data[name] = pd.read_csv(file_path)

    print(
        f"Loaded {name}: "
        f"{len(loaded_data[name]):,} rows"
    )

raw = loaded_data

print("\nAll five datasets loaded successfully.")

Loaded registration: 1,898 rows
Loaded behaviour: 18,322 rows
Loaded funding: 1,893 rows
Loaded marketing: 8,332 rows
Loaded metadata: 11,475 rows

All five datasets loaded successfully.


In [25]:
inventory = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "exact_duplicates": int(df.duplicated().sum()),
        "unique_customers": (
            df["customer_id"].nunique()
            if "customer_id" in df.columns
            else np.nan
        )
    }
    for name, df in raw.items()
])

display(inventory)

class_counts = (
    raw["funding"]["activation_flag"]
    .value_counts(dropna=False)
)

class_summary = pd.DataFrame({
    "customers": class_counts,
    "percentage": (
        class_counts / len(raw["funding"]) * 100
    ).round(2)
})

display(class_summary)

inventory.to_csv(
    os.path.join(results_folder, "source_inventory.csv"),
    index=False
)

# Record file hashes so the input version can be checked.
input_hashes = {}

for name, filename in files.items():
    with open(os.path.join(data_folder, filename), "rb") as file:
        input_hashes[name] = hashlib.sha256(
            file.read()
        ).hexdigest()

with open(
    os.path.join(results_folder, "input_sha256.json"), "w"
) as file:
    json.dump(input_hashes, file, indent=2)

,dataset,rows,columns,exact_duplicates,unique_customers
0,registration,1898,13,5,1893.0
1,behaviour,18322,11,0,1893.0
2,funding,1893,11,0,1893.0
3,marketing,8332,10,0,1893.0
4,metadata,11475,5,4,NaN


,customers,percentage
activation_flag,,
False,1085,57.32
True,808,42.68


In [26]:
for name, df in raw.items():
    print(f"\n{name.upper()}")
    print("Columns:", df.columns.tolist())
    display(df.head(3))


REGISTRATION
Columns: ['customer_id', 'registration_date', 'registration_channel', 'acquisition_channel', 'kyc_start_date', 'kyc_completion_date', 'kyc_status', 'kyc_steps_completed', 'onboarding_abandonment_step', 'age', 'region', 'investment_goal', 'risk_appetite']


,customer_id,registration_date,registration_channel,acquisition_channel,kyc_start_date,kyc_completion_date,kyc_status,kyc_steps_completed,onboarding_abandonment_step,age,region,investment_goal,risk_appetite
0,CUST1000000,2025-06-07 17:56:56,Mobile app,Organic search,2025-06-07 22:59:11,2025-06-07 23:39:30,Completed,6,NaN,37.0,Western Cape,Retirement,Moderate
1,CUST1000001,2025-03-22 20:40:55,Mobile app,Referral,2025-03-22 22:56:22,2025-03-23 03:26:48,Completed,6,NaN,18.0,Limpopo,Long-term growth,Moderate
2,CUST1000002,2025-03-12 23:17:29,Web,Organic search,2025-03-13 05:45:28,2025-03-13 07:50:44,Completed,6,NaN,32.0,Gauteng,Savings,Conservative



BEHAVIOUR
Columns: ['session_id', 'customer_id', 'session_start', 'session_duration', 'platform', 'screen_viewed', 'screen_views', 'feature_used', 'onboarding_step', 'abandonment_flag', 'event_timestamp']


,session_id,customer_id,session_start,session_duration,platform,screen_viewed,screen_views,feature_used,onboarding_step,abandonment_flag,event_timestamp
0,SES96368,CUST1000654,2025-12-16 16:21:58,8.0,Web - Mobile,Registration,7,Comparison tool,Risk profile questionnaire,False,2025-12-16 16:27:12
1,SES91185,CUST1000125,2026-07-21 23:50:14,12.3,Web - Mobile,Deposit,7,Education,Proof of address,False,2026-07-21 23:53:41
2,SES91796,CUST1000184,2025-05-15 05:26:53,6.6,Tablet,Deposit confirmation,4,Deposit,Risk profile questionnaire,False,2025-05-15 05:38:14



FUNDING
Columns: ['customer_id', 'first_deposit_date', 'first_deposit_amount', 'deposit_count', 'total_deposited', 'withdrawal_count', 'account_balance', 'balance_as_at', 'activation_flag', 'activation_date', 'days_to_first_deposit']


,customer_id,first_deposit_date,first_deposit_amount,deposit_count,total_deposited,withdrawal_count,account_balance,balance_as_at,activation_flag,activation_date,days_to_first_deposit
0,CUST1000000,2025-06-11 11:52:39,1347.17,5,3500.28,2,3436.94,2026-09-14 09:44:03,True,2025-06-11 11:52:39,3.0
1,CUST1000001,NaN,NaN,0,0.00,0,0.00,2026-09-14 09:44:03,False,NaN,NaN
2,CUST1000002,2025-03-15 01:12:24,666.16,1,666.16,1,507.65,2026-09-14 09:44:03,True,2025-03-15 01:12:24,2.0



MARKETING
Columns: ['customer_id', 'campaign_id', 'campaign_channel', 'campaign_date', 'message_type', 'message_sent', 'message_opened', 'link_clicked', 'nudge_response', 'acquisition_cost']


,customer_id,campaign_id,campaign_channel,campaign_date,message_type,message_sent,message_opened,link_clicked,nudge_response,acquisition_cost
0,CUST1000914,CMP202506045,Push notification,2025-06-01 04:20:06,Promotional offer,True,True,True,False,27.46
1,CUST1001147,CMP202409098,SMS,2024-09-16 02:06:43,Onboarding reminder,True,NaN,NaN,NaN,25.42
2,CUST1001860,CMP202410058,Push notification,2024-10-27 11:36:55,Educational tip,True,True,True,False,37.67



METADATA
Columns: ['record_timestamp', 'source_system', 'record_status', 'missingness_reason', 'data_extraction_date']


,record_timestamp,source_system,record_status,missingness_reason,data_extraction_date
0,2024-10-09 01:15:50,App analytics,Active,Not Applicable,2026-09-14 09:44:03
1,2026-04-24 02:29:14,Marketing automation,Active,Not Applicable,2026-09-14 09:44:03
2,2025-02-06 15:34:19,App analytics,Active,Not Applicable,2026-09-14 09:44:03
